# Lektion 18 (fortsättning): Kvitton som bevisar att en *människa* godkände åtgärden

Lektionen bevisar vad **agenten** gjorde och vad **grinden** beslutade. Den här anteckningsboken lägger till den saknade halvan: bevis på att en **namngiven människa** godkände **exakt** åtgärd – en separat, mänskligt hållbar signatur över hela den kanoniska åtgärden, verifierad offline.

Båda artefakterna här använder **samma kuvertform som lektionens kvitton**: en platt nyttolast med ett `type`-fält, signerad med Ed25519 över SHA-256 av de kanoniska nyttolastbitarna, med ett strukturerat `signature`-objekt bifogat (och uteslutet från de signerade bitarna). Godkännandekvittot är en ny `type` (`human.approval.v1`) vid sidan av åtgärdstypen, så en `verify_chain` täcker båda artefakttyperna med samma kodväg som du byggde i huvudanteckningsboken. Detta är också formen på samsignaturvägen i Internet-Draft som lektionen följer (draft-farley-acta-signed-receipts).

En avsiktlig uppgradering jämfört med demo-verifieraren i huvudanteckningsboken: verifieraren här löser `signature.key_id` mot ett **pinnat nyckelregister** istället för att lita på en offentlig nyckel som bärs med i kvittot. Det är den produktionsposition som lektionens egen checklist rekommenderar ("publicera verifieringsnyckeln"), och det är vad som gör förfalskning till ett avslag istället för en egen-nyckel-omgåelse.

Reglen som den här anteckningsboken lär ut: **en signerad godkännande är inte auktoritet i sig själv.** Auktoritet finns bara om godkännandekvittot och åtgärdskvittot fortfarande är bundna till samma kanoniska åtgärd vid utförandet, under en policyversion, nyckel och giltighetstid som fortfarande är aktuell, och ett godkännande som inte redan har förbrukats. Varje fel avvisas med en **annan orsak**, så att du kan skilja på *auktoriteten blev föråldrad* från *den utförda åtgärden ändrades*.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## Den exakta åtgärden

Enheten för godkännande är **det kanoniska åtgärdsobjektet** — inte en vag etikett som "godkänn återbetalning," utan den precisa, fullständigt specificerade åtgärden. Att skriva över hela objektet (och härleda en digest från det) är vad som låter oss bevisa senare att människan godkände *detta* och inget annat.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## Ett kuvert, två myndigheter

Varje kvitto är lektionens kuvert: en platt nyttolast med ett `type`-fält, plus ett `signature`-objekt (`alg`, `sig`, `key_id`) som **inte** är en del av de signerade bytena. `verify_envelope` är den gemensamma strukturella + signaturkontrollen för båda kvitto-typerna; vilken **pinned key registry** den löser `signature.key_id` mot är det som håller myndigheterna åtskilda:

- **approval receipt** (`human.approval.v1`) — namngiven godkännare, hela den kanoniska handlingen **och dess digest**, `policy_version`, utfärdande- och utgångstidsstämplar. Engångskonsumtion spåras på kedjenivå.
- **action receipt** (`agent.action.v1`) — agentidentitet, `run_id`, samma kanoniska handlings **digest**, exekveringsresultat + tidsstämpel, och `parent_approval_ref`: `receipt_hash` av godkännandet, samma konvention som `previous_receipt_hash` i lektionens kedja.

Det gemensamma `action_digest`-fältet är den sammanfogning som bindningen beror på. `key_id` finns i signaturobjektet endast som en uppslagsledtråd: att peka om den till en annan pinned key gör att signaturkontrollen misslyckas, så den ger inget.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: där bindningen faktiskt avgörs

`verify_chain` är **inte** en bekvämlighetsomslag över två signaturkontroller. Det är den enda platsen där den gemensamma kanoniska `action_digest`, policyn/nyckelns/godkännandets **aktuella giltighet** och godkännandets **engångskonsumtion** kontrolleras tillsammans, mot den åtgärd som utförs *just nu*.

Varje fel nekas med en **distinkt orsak**, så att en läsare av nekandet kan avgöra om auktoriteten blev föråldrad (policy flyttades, nyckel roterades, godkännandet gick ut, godkännandet konsumerades) eller om den utförda åtgärden ändrades under en fortfarande giltig godkännande (digest-substitution).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## Vad bindningen fångar

Varje fall nedan misslyckas **stängt** med en **distinkt anledning**. Den första blocket är den klassiska uppsättningen (manipulation, förvirrad ställföreträdare, återspelning, förfalskning på endera auktoritet, felaktig indata). Det andra blocket är paret som gör egenskapen verklig snarare än påstådd:

- **föråldrad auktoritet** — signaturen är fortfarande giltig, men policyns version har ändrats, godkännarnyckeln roterades bort från den fastställda registret, eller godkännandet löpte ut före genomförandet;
- **digest-utbyte** — ett giltigt signerat åtgärdsbevis vars `parent_approval_ref` pekar på ett *verkligt* godkännande, men det godkännandets kanoniska åtgärdsdigest stämmer inte överens med den åtgärd som faktiskt utförs.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## Vad detta bevisar — och vad det inte gör

**Bevisar:** att en namngiven människa godkände *denna exakta kanoniska åtgärd* (full åtgärd + digest, signerad med en nyckel löst från ett fixerat register), och att agenten utförde *precis den godkända åtgärden* (samma digest, kvitto bundet till godkännandet via `receipt_hash`, lektionens egen kedjekonvention) — samtidigt som godkännandets policysversion, nyckel och utgång var aktuella, exakt en gång. Om någon av sidorna ändras, stängs kedjan, och avvisningsorsaken berättar för dig **vilken** egenskap som bröts: föråldrad auktoritet vs. en ändrad åtgärd.

**Bevisar inte:** att godkännande-UI:et visade människan vad de trodde att de signerade (WYSIWYS är ett eget problem), att nyckeln inte tvingades fram eller stals före rotation, eller att följdeffekter motsvarade åtgärden. Signerat ≠ auktoriserat: en giltig signatur över en föråldrad policy, en roterad nyckel, ett utgånget fönster eller en annan digest ger ingenting här.

De två kvittoslagen delar lektionens omslag och en `verify_chain`-kodväg med flit: bindningen du byggde för åtgärdskvitton i huvudanteckningsboken är samma kod som kontrollerar människans godkännande. Ett verifieringskontrakt, separata fixerade auktoriteter, förenade av den kanoniska åtgärdsdigesten och inget annat.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfriskrivning**:
Detta dokument har översatts med hjälp av AI-översättningstjänsten [Co-op Translator](https://github.com/Azure/co-op-translator). Även om vi strävar efter noggrannhet, var vänlig notera att automatiska översättningar kan innehålla fel eller brister. Det ursprungliga dokumentet på dess modersmål bör betraktas som den auktoritativa källan. För kritisk information rekommenderas professionell mänsklig översättning. Vi ansvarar inte för några missförstånd eller feltolkningar som uppstår till följd av användningen av denna översättning.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
